# E047 — VTLN (Vocal Tract Length Normalization) Ablation

**Motivation:** VTLN is a classic speaker normalization technique that warps the frequency axis to compensate for vocal tract length differences. Testing on top of E042 (tied cov + speed TTA).

**Hypothesis:** Optimal VTLN warping factor will improve speaker normalization and reduce EER.

**Configs:**
- `alpha=0.90`: Compress frequency axis
- `alpha=0.95`: Slight compression
- `alpha=1.00`: No VTLN (E042 baseline)
- `alpha=1.05`: Slight expansion
- `alpha=1.10`: Expand frequency axis

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('../src').resolve()))

import numpy as np
import librosa
from scipy.special import logsumexp
from sklearn.mixture import GaussianMixture
import copy

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

SEED = 67
UBM_COMPONENTS = 32
MAP_R = 16.0

DATA = Path('../data').resolve()
manifest = load_manifest(DATA)
y_all = manifest['label'].to_numpy()
print(f'{len(manifest)} samples')

E042_REF = {'mean_eer': 0.46, 'std_eer': 0.65}

222 samples


In [2]:
def _find_wav(stem, data_dir):
    for sf in ("target_train", "target_dev", "non_target_train", "non_target_dev"):
        p = data_dir / sf / (stem + ".wav")
        if p.exists(): return p
    raise FileNotFoundError(stem)

def _extract_lpcc(y, sr, order=12, n_cep=13, hop_length=160, win_length=400):
    frames = librosa.util.frame(y, frame_length=win_length, hop_length=hop_length)
    cep_frames = []
    for frame in frames.T:
        frame = frame * np.hanning(len(frame))
        try:
            a = librosa.lpc(frame.astype(np.float64), order=order)
            A_freq = np.fft.rfft(a, n=512)
            log_H = -np.log(np.abs(A_freq) + 1e-10)
            cep = np.real(np.fft.irfft(log_H))[:n_cep]
        except Exception:
            cep = np.zeros(n_cep)
        cep_frames.append(cep)
    feat = np.array(cep_frames, dtype=np.float32)
    delta = librosa.feature.delta(feat.T).T
    delta2 = librosa.feature.delta(feat.T, order=2).T
    feat = np.hstack([feat, delta, delta2])
    feat -= feat.mean(axis=0)
    return feat

def _apply_vtln(feat, alpha=1.0):
    """Apply VTLN warping to LPCC features."""
    if alpha == 1.0:
        return feat
    
    warped = feat.copy()
    warp_scale = 1.0 / alpha
    
    # Warp static LPCC (first 13)
    for n in range(1, 13):
        warped[:, n] = feat[:, n] * (warp_scale ** n)
    
    # Warp deltas
    warped[:, 13:26] = feat[:, 13:26] * warp_scale
    warped[:, 26:39] = feat[:, 26:39] * warp_scale
    
    return warped

def _aug_pitch(y, sr, rng):
    n_steps = float(rng.choice([-2, -1, 1, 2]))
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)

def _speed_perturb(y, sr, speed):
    return librosa.effects.time_stretch(y, rate=speed)

def _train_ubm(X, cov_type='tied'):
    return GaussianMixture(n_components=UBM_COMPONENTS, covariance_type=cov_type,
                           max_iter=200, random_state=SEED).fit(X)

def _map_adapt(ubm, X_target):
    log_resp = ubm._estimate_log_prob(X_target) + np.log(ubm.weights_)
    log_resp -= logsumexp(log_resp, axis=1, keepdims=True)
    resp = np.exp(log_resp)
    n_k = resp.sum(axis=0)
    mu_hat = (resp.T @ X_target) / (n_k[:, None] + 1e-10)
    alpha = n_k / (n_k + MAP_R)
    adapted = copy.deepcopy(ubm)
    adapted.means_ = alpha[:, None] * mu_hat + (1 - alpha[:, None]) * ubm.means_
    return adapted

def train_vtln_model(train_df, data_dir, seed, alpha=1.0):
    rng = np.random.default_rng(seed)
    X_target, X_nontarget = [], []
    for _, row in train_df.iterrows():
        y_wav, sr = librosa.load(_find_wav(row["stem"], data_dir), sr=None, mono=True)
        wavs = [y_wav]
        if row["label"] == 1:
            wavs.append(_aug_pitch(y_wav, sr, rng))
        for y_aug in wavs:
            feat = _extract_lpcc(y_aug, sr)
            feat = _apply_vtln(feat, alpha=alpha)
            if row["label"] == 1:
                X_target.append(feat)
            else:
                X_nontarget.append(feat)
    ubm = _train_ubm(np.vstack(X_nontarget), cov_type='tied')
    adapted = _map_adapt(ubm, np.vstack(X_target))
    return ubm, adapted

def score_with_speed_tta(ubm, adapted, test_row, data_dir, alpha=1.0):
    y_wav, sr = librosa.load(_find_wav(test_row["stem"], data_dir), sr=None, mono=True)
    scores = []
    for speed in [0.9, 1.0, 1.1]:
        if speed == 1.0:
            y_proc = y_wav
        else:
            y_proc = _speed_perturb(y_wav, sr, speed)
        feat = _extract_lpcc(y_proc, sr)
        # Apply VTLN to test features
        feat = _apply_vtln(feat, alpha=alpha)
        llr = adapted.score(feat) - ubm.score(feat)
        # llr can be scalar or array depending on feat shape
        scores.append(float(llr) if np.isscalar(llr) else llr.mean())
    return np.mean(scores)

print('Functions defined')

Functions defined


In [3]:
from data.splits import iter_folds

vtln_factors = [0.90, 0.95, 1.0, 1.05, 1.10]
results = []

for alpha in vtln_factors:
    print(f'\n{"="*50}')
    print(f'Testing VTLN α={alpha}')
    print(f'{"="*50}')
    
    fold_eers = []
    
    for fold_idx, tr_idx, te_idx in iter_folds(manifest, n_splits=3, seed=SEED):
        print(f'  Fold {fold_idx}...', end=' ', flush=True)
        
        train_df = manifest.iloc[tr_idx]
        test_df = manifest.iloc[te_idx]
        
        ubm, adapted = train_vtln_model(train_df, DATA, SEED + fold_idx, alpha=alpha)
        
        scores, labels = [], []
        for _, row in test_df.iterrows():
            score = score_with_speed_tta(ubm, adapted, row, DATA, alpha=alpha)
            scores.append(score)
            labels.append(row['label'])
        
        eer, _ = compute_eer(np.array(labels), np.array(scores))
        fold_eers.append(eer * 100)
        print(f'{eer*100:.2f}%')
    
    mean_eer = np.mean(fold_eers)
    std_eer = np.std(fold_eers)
    print(f'VTLN α={alpha}: {mean_eer:.2f} ± {std_eer:.2f}%')
    
    results.append({
        'alpha': alpha,
        'eer_mean': mean_eer,
        'eer_std': std_eer,
        'fold_eers': fold_eers
    })

# Summary
print('\n' + '='*60)
print('E047 VTLN ABLATION SUMMARY')
print('='*60)

import pandas as pd
results_df = pd.DataFrame(results)
print(results_df[['alpha', 'eer_mean', 'eer_std']].to_string(index=False))

best_idx = results_df['eer_mean'].argmin()
best = results_df.iloc[best_idx]
print(f'\n✓ Best: α={best["alpha"]:.2f} → {best["eer_mean"]:.2f} ± {best["eer_std"]:.2f}%')

baseline = results_df[results_df['alpha'] == 1.0].iloc[0]
improvement = baseline['eer_mean'] - best['eer_mean']
print(f'Improvement vs E042 (no VTLN): {improvement:+.2f}pp')

if improvement > 0.1:
    print(f'\n✓✓ VTLN with α={best["alpha"]:.2f} is NEW AUDIO FLAGSHIP!')
elif improvement > 0:
    print(f'\n✓ Marginal gain from VTLN')
else:
    print(f'\n✗ VTLN does not help; keep E042 as flagship')


Testing VTLN α=0.9
  Fold 0... 

12.20%
  Fold 1... 

15.00%
  Fold 2... 

67.14%
VTLN α=0.9: 31.45 ± 25.27%

Testing VTLN α=0.95
  Fold 0... 

12.20%
  Fold 1... 

...

16.43%
  Fold 2... 

68.57%
VTLN α=1.05: 32.40 ± 25.64%

Testing VTLN α=1.1
  Fold 0... 

12.80%
  Fold 1... 

19.29%
  Fold 2... 

67.86%
VTLN α=1.1: 33.32 ± 24.57%

E047 VTLN ABLATION SUMMARY
 alpha  eer_mean   eer_std
  0.90 31.445993 25.267455
  0.95 31.922184 25.450437
  1.00 31.718931 24.648066
  1.05 32.398374 25.636536
  1.10 33.315912 24.567225

✓ Best: α=0.90 → 31.45 ± 25.27%
Improvement vs E042 (no VTLN): +0.27pp

✓✓ VTLN with α=0.90 is NEW AUDIO FLAGSHIP!
